In [13]:
#load in the dataset
import pandas as pd

df = pd.read_csv('food_ingredients_and_allergens.csv')
df_2 = pd.read_csv('allergen dataset.csv')

# collapse all labels except class labels into one 'text' feature
#from sklearn.datasets import make_multilabel_classification

#multilabeller needs a set list of labels. i think i should go with the 14 allergens. Some of these allergen names need to be normalised.
allowed = {"celery", "gluten", "crustacean", "eggs", "fish",
           "lupin", "dairy", "mollusc", "mustard", "peanuts",
           "sesame", "soy", "sulphur dioxide", "sulphites", "tree nuts"}

#create a custom dictionary of allergen names
allergen_dict = {

    "ghee" : "dairy",
    "milk" : "dairy",
    "dairy" : "dairy",

    "shellfish" : "crustacean",

    "mustard" : "mustard",

    "soy" : "soy",
    "soybean" : "soy",
    "soybeans" : "soy",

    "tree nut" : "tree nuts",
    "nuts" : "tree nuts",
    "almonds" : "tree nuts",
    "walnuts" : "tree nuts",
    "pine nuts" : "tree nuts",

    "fish" : "fish",

    "wheat" : "gluten",
    "gluten" : "gluten",
    
    "celery" : "celery",

    "peanut" : "peanuts",
    "peanuts" : "peanuts",

    "eggs" : "eggs",
    "egg" : "eggs",

    "lupine" : "lupin",

}

def normalise_allergens(allergen_list):
    normalised = set() #from each row, the set of unique allergens
    allergen_list = str(allergen_list)
    allergen_list = [a.strip() for a in allergen_list.split(",")]
    #print("allergen_list: ", allergen_list)

    for allergen in allergen_list:
        allergen = allergen.lower().strip()
        #print(allergen)

        # Step 1: map synonyms
        if allergen in allergen_dict:
            #print(allergen, "is in dict")
            #print("translation: ", allergen_dict[allergen])
            normalised.add(allergen_dict[allergen]) #translate
            continue

        # Step 2: keep only if already canonical
        if allergen in allowed:
            normalised.add(allergen) #filter
    
    return list(normalised)

x = []
#x = df['Allergens'].apply(normalise_allergens)

df['Allergens'] = df['Allergens'].apply(normalise_allergens)
df_2['allergy'] = df_2['allergy'].apply(normalise_allergens)

#display(df)

#create a new dataframe

data = {
    'Food' : [],
    'Allergens' : []
}

labels = ['Food Product', 'Main Ingredient', 'Sweetener', 'Fat/Oil', 'Seasoning']

for index, row in df.iterrows():
    food_list = ""
    for label in labels:
        if(type(row[str(label)]) == str):
            food_list = food_list + (row[str(label)]) + " "
    data["Food"].append(food_list)
    data["Allergens"].append(row["Allergens"])

#for index, row in df_2.iterrows():
#    data["Food"].append(row['ingredient'])
#    data["Allergens"].append(row["allergy"])

display(pd.DataFrame(data))


,Food,Allergens
0,Almond Cookies Almonds Sugar Butter Flour,"[gluten, dairy, tree nuts]"
1,Almond Cookies Almonds Sugar Butter Flour,"[gluten, dairy, tree nuts]"
2,Chicken Noodle Soup Chicken broth Salt,"[gluten, celery]"
3,Chicken Noodle Soup Chicken broth Salt,"[gluten, celery]"
4,Cheddar Cheese Cheese Salt,[dairy]
...,...,...
394,"Lemon Bars Lemon juice Sugar Butter Flour, eggs","[gluten, dairy, eggs]"
395,Pecan Pie Pecans Sugar Butter Corn syrup,"[gluten, dairy, tree nuts]"
396,"Zucchini Bread Zucchini Sugar Butter Cinnamon,...","[gluten, dairy, tree nuts]"
397,"Banana Bread Bananas Sugar Butter Cinnamon, nuts","[gluten, dairy, tree nuts]"


In [14]:
#encode for multi labelling
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# Sample multi-label data
y = data["Allergens"]

# Initialize and fit MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y)

food = data["Food"]

print(y_encoded)
print(food)
print(mlb.classes_)
print(allowed)



[[0 0 1 ... 0 0 1]
 [0 0 1 ... 0 0 1]
 [1 0 0 ... 0 0 0]
 ...
 [0 0 1 ... 0 0 1]
 [0 0 1 ... 0 0 1]
 [0 0 1 ... 0 0 0]]
['Almond Cookies Almonds Sugar Butter Flour ', 'Almond Cookies Almonds Sugar Butter Flour ', 'Chicken Noodle Soup Chicken broth Salt ', 'Chicken Noodle Soup Chicken broth Salt ', 'Cheddar Cheese Cheese Salt ', 'Ranch Dressing Buttermilk Sugar Vegetable oil Garlic, herbs ', 'Caramel Popcorn Popcorn Sugar Butter Salt ', 'Caesar Salad Romaine lettuce Olive oil Parmesan cheese ', 'Caesar Wrap Grilled chicken Caesar dressing Lettuce, Parmesan cheese ', 'Strawberry Smoothie Strawberries Honey Yogurt (milk, cultures) ', 'Cheese Pizza Cheese Tomato sauce ', 'Margherita Pizza Cheese Tomato sauce, basil ', 'Mashed Potatoes Potatoes Butter Salt, Pepper ', 'Greek Yogurt Yogurt (milk, cultures) ', 'Caesar Salad Wrap Grilled chicken Caesar dressing Lettuce, Parmesan cheese ', 'Caprese Salad Tomatoes Olive oil Mozzarella cheese, basil ', 'Berry Smoothie Mixed berries Sugar Yogurt (m

In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.multioutput import MultiOutputClassifier #Using this we can retrofit any classifier to be a multi labeller. 
from sklearn.linear_model import LogisticRegression

# initializing TfidfVectorizer 
vetorizar = TfidfVectorizer()
# fitting the tf-idf on the given data
X = vetorizar.fit_transform(food)

print(X.shape)
print(y_encoded.shape)

# Splitting the dataset into train and test sets
X_train, y_train, X_test, y_test = iterative_train_test_split(X, y_encoded, test_size=0.2)

# Creating the MultiOutput Classifier with Logistic Regression as the base estimator
classifier = MultiOutputClassifier(LogisticRegression(max_iter=1000))

# Fitting the classifier on the training data
classifier.fit(X_train, y_train)

# Making predictions on the test set
predictions = classifier.predict(X_test)

# Evaluate the model, for example, using accuracy score
accuracy = classifier.score(X_test, y_test)
print("Accuracy:", classifier.m)



(399, 391)
(399, 10)
Accuracy: 0.6375


In [ ]:
new_sentences = ["Almond Cookies"]
new_sentence_tfidf = vetorizar.transform(new_sentences)
probas = classifier.predict_proba(new_sentence_tfidf)

predicted_sentences = classifier.predict(new_sentence_tfidf)
#print(predicted_sentences)
#print(mlb.classes_)

decoded = [mlb.classes_[i] for i, val in enumerate(predicted_sentences[0]) if (val == 1).any()]
print(decoded)



['dairy']
